# Umgang mit kategorialen Variablen



## Lernziele



Am Ende dieses Notebooks solltest du Folgendes können:
- Herausfinden, welche Variablen kategoriale Variablen (categorical variables) sind.
- Eine Variable mit Label Encoding (label encoder) und Dummy-Variablen (dummy variables) transformieren.
- Multikollinearität (multicollinearity) erklären und sie bei der Verwendung von Dummy-Variablen vermeiden.


Um all das herauszufinden, brauchen wir einen Beispieldatensatz. Wir bleiben bei den Auto-Daten, die wir für die multiple lineare Regression verwendet haben. 

In [ ]:
# Benötigte Bibliotheken importieren
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Daten importieren
data = pd.read_csv("data/cars_multivariate.csv")
data.head()

In [ ]:
# Datentyp jeder Spalte prüfen
data.info()

Außer `car name` scheint jede andere Spalte numerische Werte zu enthalten. Sie könnten auch Kandidaten sein, um die abhängige Variable `mpg` (Meilen pro Gallone) zu beschreiben. Auf den ersten Blick scheint es, als hätten wir keine kategorialen Variablen.



## Was sind kategoriale Variablen?



Normalerweise kennen wir die Art der Daten und wissen daher, welche Variablen kategorial oder numerisch sind. Aber auch wenn wir die Daten nicht kennen, können wir herausfinden, ob die Variablen kategorial oder numerisch sein könnten. Schauen wir uns jetzt die Spalte `origin` genauer an.

In [ ]:
# Deskriptive Statistik für die Spalte origin
print(data["origin"].describe())

In [ ]:
# Anzahl der eindeutigen Werte in der Spalte origin
print(data["origin"].nunique())

In [ ]:
# Eindeutige Werte in der Spalte origin
print(data["origin"].unique())

Die Werte reichen von 1 bis 3, und tatsächlich kommen im Datensatz nur die Werte 1, 2 und 3 vor! Es stellt sich heraus, dass "origin" eine sogenannte **kategoriale** Variable ist. Sie steht nicht für eine kontinuierliche Zahl, sondern bezieht sich auf einen Ort: 1 könnte zum Beispiel für die USA stehen, 2 für Europa, 3 für Asien (Hinweis: Für diesen Datensatz ist die tatsächliche Bedeutung nicht offengelegt).

Kategoriale Variablen sind also genau das, wonach sie klingen: Sie stehen für Kategorien, nicht für numerische Features. Beachte, dass solche Features oft als Textwerte gespeichert werden, die verschiedene Ausprägungen (levels) der Beobachtungen darstellen, auch wenn das hier nicht der Fall ist.



## Kategoriale Variablen identifizieren



Da kategoriale Variablen auf eine besondere Weise behandelt werden müssen, wie du später sehen wirst, musst du sicherstellen, dass du erkennst, welche Variablen kategorial sind.



In manchen Fällen ist das Erkennen einfach (z. B. wenn sie als Strings gespeichert sind), in anderen Fällen sind sie numerisch, und die Tatsache, dass sie kategorial sind, ist nicht immer sofort ersichtlich.



Beachte, dass das nicht trivial sein kann. Als Erstes kannst du die Methoden `.describe()` und `.info()` verwenden. `.describe()` gibt dir Informationen über die Datentypen (wie Strings, Integer usw.), aber selbst dann könnten kontinuierliche Variablen als Strings importiert worden sein. Deshalb ist es sehr wichtig, dass du dir deine Daten wirklich anschaust. 

In [ ]:
# Diagramme erstellen, um kategoriale Features zu identifizieren
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 3))

for xcol, ax in zip(["acceleration", "displacement", "horsepower", "weight"], axes):
    data.plot(kind="scatter", x=xcol, y="mpg", ax=ax, alpha=0.4, color="b")

In [ ]:
# Diagramme erstellen, um kategoriale Features zu identifizieren
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 3))

for xcol, ax in zip(["cylinders", "model_year", "origin"], axes):
    data.plot(kind="scatter", x=xcol, y="mpg", ax=ax, alpha=0.4, color="b")

Beachte den strukturellen Unterschied zwischen der oberen und der unteren Gruppe von Diagrammen. Du siehst, dass die Struktur sehr unterschiedlich aussieht: Statt einer ziemlich homogenen "Wolke" erzeugen kategoriale Variablen senkrechte Linien bei diskreten Werten. Ein weiterer Diagrammtyp, den man sich ansehen kann, ist das Histogramm.

In [ ]:
data.hist(figsize=(8, 8));

Und die Anzahl der eindeutigen Werte:

In [ ]:
data[["cylinders", "model_year", "origin"]].nunique()

## Kategoriale Variablen transformieren

Wenn du kategoriale Variablen in Regressionsmodellen verwenden willst, müssen sie transformiert werden. Dafür gibt es zwei Ansätze:
* Label Encoding durchführen
* Dummy-Variablen erstellen / One-Hot-Encoding

### Label Encoding

Wir veranschaulichen Label Encoding und das Erstellen von Dummies mit der folgenden pandas Series mit 3 Kategorien: "USA", "EU" und "ASIA".

In [ ]:
origin = ["USA", "EU", "EU", "ASIA", "USA", "EU", "EU", "ASIA", "ASIA", "USA"]
origin_series = pd.Series(origin)

In [ ]:
origin_df = pd.DataFrame({"origin": origin})

Jetzt willst du sicherstellen, dass Python diese Strings als Kategorien erkennt. Das geht so:

In [ ]:
cat_origin = origin_series.astype("category")
cat_origin

Beachte, dass der `dtype` (also der Datentyp) hier `category` ist und die drei Kategorien erkannt werden.

Manchmal willst du deine Labels als Zahlen darstellen. Das nennt man Label Encoding.

Du führst das Label Encoding so durch, dass die numerischen Labels immer zwischen 0 und (number_of_categories)-1 liegen. Dafür gibt es mehrere Möglichkeiten, eine davon ist `.cat.codes`

In [ ]:
cat_origin.cat.codes

Eine andere Möglichkeit ist der `OrdinalEncoder` von scikit-learn:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ord_make = OrdinalEncoder()

origin_encoded = ord_make.fit_transform(origin_df[["origin"]])

In [ ]:
origin_encoded

Beachte: Während `.cat.codes` nur bei Variablen verwendet werden kann, die mit `.astype(category)` transformiert wurden, ist das für den `OrdinalEncoder` keine Voraussetzung.

### Dummy-Variablen erstellen

Eine weitere Möglichkeit, kategoriale Variablen zu transformieren, ist One-Hot-Encoding bzw. "Dummy-Variablen". Die Idee ist, jede Kategorie in eine neue Spalte umzuwandeln und der Spalte eine 1 oder 0 zuzuweisen. Es gibt mehrere Bibliotheken, die One-Hot-Encoding unterstützen. Schauen wir uns zwei davon an:

In [ ]:
# Dummy-Variablen mit pandas .get_dummies() erstellen
pd.get_dummies(cat_origin)

Sieh dir an, wie der Label-Name zum Spaltennamen geworden ist! Eine weitere Methode ist der `LabelBinarizer` in scikit-learn. 

In [ ]:
from sklearn.preprocessing import LabelBinarizer

lb = LabelBinarizer()
origin_dummies = lb.fit_transform(cat_origin)

# Das musst du wieder in einen DataFrame umwandeln
origin_dum_df = pd.DataFrame(origin_dummies, columns=lb.classes_)
origin_dum_df

Der Vorteil von Dummies ist, dass deine numerischen Werte nicht als kontinuierlich fehlinterpretiert werden können, egal welchen Algorithmus du verwendest. Für die Zukunft ist es wichtig zu wissen, dass für die lineare Regression (und die meisten anderen Algorithmen in scikit-learn) **One-Hot-Encoding erforderlich ist**, wenn du kategoriale Variablen in ein Regressionsmodell aufnimmst!

## Die Dummy-Variablen-Falle (Dummy Variable Trap)



Aufgrund der Art, wie Dummy-Variablen erstellt werden, lässt sich eine Variable aus allen anderen vorhersagen. Das nennt man perfekte **Multikollinearität** (multicollinearity), und sie kann bei der Regression ein Problem sein. Die Grundidee hinter perfekter Multikollinearität ist, dass du den Wert einer Variable mit einer Kombination der anderen Variablen *perfekt* vorhersagen kannst. Falls das nicht ganz klar ist, geh noch einmal zu den One-Hot-codierten origin-Daten von oben zurück:

In [ ]:
# Dummy-Variablen mit pandas .get_dummies() erstellen
trap_df = pd.get_dummies(cat_origin)
trap_df.head()

Weil du für jede Herkunft (origin) eine Dummy-Variable erstellt hast, kannst du jetzt jede einzelne origin-Dummy-Variable mit den Informationen aus allen anderen vorhersagen. OK, das klingt vielleicht eher nach einem Zungenbrecher als nach einer Erklärung, also konzentrier dich erst einmal auf die Spalte ASIA. Du kannst diese Spalte perfekt vorhersagen, indem du die Werte der Spalten EU und USA addierst und die Summe dann von 1 abziehst, wie unten gezeigt:

In [ ]:
# Spalte ASIA aus EU und USA vorhersagen
predicted_asia = 1 - (trap_df["EU"] + trap_df["USA"])
predicted_asia.to_frame(name="Predicted_ASIA")

EU und USA lassen sich auf ähnliche Weise vorhersagen, das kannst du selbst ausarbeiten.

Du fragst dich wahrscheinlich, warum das für die Regression ein Problem ist. Zur Erinnerung: Die Koeffizienten, die aus einem Regressionsmodell abgeleitet werden, werden für Vorhersagen verwendet. In einer multiplen linearen Regression stellen die Koeffizienten die durchschnittliche Veränderung der abhängigen Variable pro Änderung der Prädiktorvariable um 1 Einheit dar, unter der Annahme, dass alle anderen Prädiktorvariablen konstant gehalten werden. Das gilt nicht mehr, wenn Prädiktorvariablen miteinander zusammenhängen, was, wie du gerade gesehen hast, automatisch passiert, wenn du Dummy-Variablen erstellst. Das ist die sogenannte **Dummy-Variablen-Falle (Dummy Variable Trap)**.

Zum Glück lässt sich die Dummy-Variablen-Falle vermeiden, indem du einfach eine der Dummy-Variablen weglässt. Das kannst du tun, indem du manuell eine Teilmenge des DataFrames auswählst oder, bequemer, indem du ```drop_first=True``` an ```get_dummies()``` übergibst: 

In [ ]:
pd.get_dummies(cat_origin, drop_first=True)

Wenn du dir den DataFrame oben genau ansiehst, wirst du feststellen, dass nicht mehr genug Information vorhanden ist, um eine der Spalten vorherzusagen. Die Multikollinearität ist also beseitigt.

Du wirst bald sehen, dass das Weglassen der ersten Variable die Interpretation der Regressionskoeffizienten beeinflusst. Die weggelassene Kategorie wird zur sogenannten **Referenzkategorie (reference category)**. Die Regressionskoeffizienten, die sich beim Fitten der verbleibenden Variablen ergeben, stellen die Veränderung *relativ* zur Referenz dar.

Du wirst außerdem sehen, dass Multikollinearität und die Dummy-Variablen-Falle in bestimmten Kontexten weniger problematisch sind und ignoriert werden können. Deshalb ist es wichtig zu verstehen, welche Modelle empfindlich auf Multikollinearität reagieren und welche nicht.

## Zurück zu unseren Auto-Daten

Wandeln wir jetzt unsere Spalten `cylinders`, `model_year` und `origin` in Dummies um und lassen jeweils die erste Variable weg.

In [ ]:
# Dummy-Variablen für kategoriale Spalten erstellen
cyl_dummies = pd.get_dummies(data["cylinders"], prefix="cyl", drop_first=True)
yr_dummies = pd.get_dummies(data["model_year"], prefix="yr", drop_first=True)
orig_dummies = pd.get_dummies(data["origin"], prefix="orig", drop_first=True)

Es könnte sein, dass die Reihenfolge der Werte in den Spalten `cylinders`, `model_year` und `origin` eine Rolle spielt. Prüf das gern mit dem adjustierten $R^2$ (adjusted $R^2$), unsere Ergebnisse deuteten darauf hin, dass die Reihenfolge keine Rolle spielt. Als Nächstes entfernen wir die ursprünglichen Spalten aus unseren Daten und fügen stattdessen die Dummy-Spalten hinzu

In [ ]:
# Ursprüngliche Spalten entfernen
data = data.drop(["cylinders", "model_year", "origin"], axis=1)

In [ ]:
# Dummy-Variablen an den DataFrame anhängen
data = pd.concat([data, cyl_dummies, yr_dummies, orig_dummies], axis=1)
data.head()

**Probier es selbst aus**

Findest du eine Kombination von Variablen (auch Dummy-Variablen), die dein letztes Modell aus Notebook 5 schlägt?



## Zusammenfassung



Du weißt jetzt,
- dass du dir deine Daten genau ansehen musst, um **kategoriale Variablen zu finden** (EDA). Dafür kannst du dir die Info und die deskriptiven Statistiken ansehen und herausfinden, wie viele eindeutige Werte jede Variable hat. Auch Streudiagramme (scatter plots) und Histogramme können helfen, kategoriale Variablen zu finden.
- wie du eine Variable mit **Label Encoding** und **Dummy-Variablen** transformierst. Für beide Wege kannst du zum Beispiel die [Preprocessing-Tools von scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features) verwenden.
- was **Multikollinearität** ist und wie du sie **bei der Verwendung von Dummy-Variablen vermeidest**.
"Multikollinearität bezeichnet eine Situation, in der mehr als zwei erklärende Variablen in einem multiplen Regressionsmodell stark linear zusammenhängen. Perfekte Multikollinearität liegt vor, wenn zum Beispiel, wie in der obigen Gleichung, die Korrelation zwischen zwei unabhängigen Variablen gleich 1 oder −1 ist." [Multicollinearity Wikipedia](https://en.wikipedia.org/wiki/Multicollinearity#:~:text=Multicollinearity%20refers%20to%20a%20situation,equal%20to%201%20or%20%E2%88%921.). Um Kollinearität bei der Verwendung von Dummy-Variablen zu vermeiden, musst du eine Dummy-Spalte weglassen.